# Bài 6: Máy tự tìm nhóm
**Mở hộp đen AI · Tập 1 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Tên bài hát và số liệu trong bài do nhóm tác giả tự đặt để luyện tập.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 1. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Danh sách 24 bài hát, mỗi bài có 3 đặc trưng
- **nhip**: số nhịp mỗi phút (BPM)
- **nang_luong**: độ sôi động, thang 1 đến 10
- **vui**: độ vui tươi, thang 1 đến 10

Chú ý: **không có cột nhãn** nào nói bài nào thuộc thể loại gì.

In [ ]:
import pandas as pd

bai_hat = pd.DataFrame({
    "ten": ["Mưa đầu mùa", "Phố cũ", "Lá vàng", "Chiều sân ga", "Ngõ nhỏ",
            "Thư không gửi", "Đèn khuya", "Hoa sữa", "Sáng thứ hai", "Đường về",
            "Tách cà phê", "Gió nồm", "Xe đạp", "Bến cũ", "Lên đồi",
            "Bước nhảy", "Mùa hè rực rỡ", "Sân khấu", "Đêm hội", "Cháy lên",
            "Tim đập nhanh", "Lễ hội", "Tốc độ", "Ánh đèn"],
    "nhip": [72, 138, 96, 120, 84, 150, 110, 76,
             128, 92, 144, 80, 116, 100, 132, 74,
             124, 98, 140, 82, 112, 146, 90, 104],
    "nang_luong": [1, 2, 1, 2, 1, 2, 1, 2,
                   5, 6, 5, 5, 6, 5, 6, 6,
                   9, 10, 9, 10, 9, 10, 9, 10],
    "vui": [2, 1, 2, 1, 2, 1, 1, 2,
            5, 5, 6, 5, 5, 6, 5, 6,
            10, 9, 9, 10, 10, 9, 9, 10],
})
print(bai_hat.head())
print("Số bài:", len(bai_hat))

### Ô 2. Gom thành 3 nhóm bằng k-means

In [ ]:
from sklearn.cluster import KMeans

X = bai_hat[["nhip", "nang_luong", "vui"]]
km = KMeans(n_clusters=3, n_init=10, random_state=0)
bai_hat["nhom"] = km.fit_predict(X)
print(bai_hat.groupby("nhom")[["nhip", "nang_luong", "vui"]].mean().round(1))

## Chặng 2: Đọc
Trả lời vào vở trước khi đi tiếp:
1. Nhìn bảng trung bình ở ô 2: ba nhóm khác nhau chủ yếu ở cột nào?
2. Một bài buồn có nhịp nhanh (ví dụ 'Phố cũ', 138 nhịp) bị xếp vào nhóm nào? Có hợp lý không?
3. Vì sao cột `nhip` lại lấn át hai cột kia? Gợi ý: so khoảng giá trị của các cột.

## Chặng 3: Sửa
### Ô 3. Đưa các cột về cùng một thang rồi gom lại

In [ ]:
from sklearn.preprocessing import StandardScaler

X_chuan = StandardScaler().fit_transform(X)   # mỗi cột: trung bình 0, lệch chuẩn 1
km2 = KMeans(n_clusters=3, n_init=10, random_state=0)
bai_hat["nhom_moi"] = km2.fit_predict(X_chuan)
print(bai_hat.groupby("nhom_moi")[["nhip", "nang_luong", "vui"]].mean().round(1))

### Ô 4. Chọn lại đặc trưng: chỉ giữ năng lượng và độ vui
Nhịp nhanh hay chậm không quyết định bài hát buồn hay vui. Con người chọn đặc trưng, không phải máy.

In [ ]:
X2 = StandardScaler().fit_transform(bai_hat[["nang_luong", "vui"]])
km3 = KMeans(n_clusters=3, n_init=10, random_state=0)
bai_hat["nhom_2"] = km3.fit_predict(X2)
print(bai_hat.groupby("nhom_2")[["nhip", "nang_luong", "vui"]].mean().round(1))
print(bai_hat.groupby("nhom_2")["ten"].apply(list).str[:4])

### Ô 5. Nên chia mấy nhóm? Thử k từ 1 đến 6

In [ ]:
for k in range(1, 7):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X2)
    print(f"k = {k}: tổng bình phương khoảng cách = {km_k.inertia_:.1f}")

### Ô 6. Gợi ý bài hát: thích bài này thì nghe thử các bài cùng nhóm

In [ ]:
thich = "Mưa đầu mùa"
nhom_cua_bai = bai_hat.loc[bai_hat["ten"] == thich, "nhom_2"].iloc[0]
goi_y = bai_hat[(bai_hat["nhom_2"] == nhom_cua_bai) & (bai_hat["ten"] != thich)]
print("Vì em thích", thich, "- thử nghe:", list(goi_y["ten"]))

### Ô 7. 'Dành cho bạn': tìm người có gu giống em nhất
Điểm từ 1 đến 5 mà bốn bạn chấm cho 5 bài. Ô trống là chưa nghe.

In [ ]:
import numpy as np

diem = pd.DataFrame({
    "Mưa đầu mùa": [5, 4, 1, 5],
    "Phố cũ":      [4, 5, 2, np.nan],
    "Bước nhảy":   [1, 2, 5, 2],
    "Đêm hội":     [2, 1, 5, 2],
    "Lá vàng":     [np.nan, 5, 1, 4],
}, index=["Lan", "Minh", "Khoa", "An"])

def khoang_cach(a, b):
    # chỉ so những bài cả hai đã nghe
    chung = diem.loc[a].notna() & diem.loc[b].notna()
    hieu = diem.loc[a, chung] - diem.loc[b, chung]
    return np.sqrt((hieu ** 2).sum())

for ban in ["Minh", "Khoa", "An"]:
    print(f"Khoảng cách gu nhạc Lan - {ban}: {khoang_cach('Lan', ban):.2f}")

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(bai_hat["nhip"], bai_hat["nang_luong"], c=bai_hat["nhom_2"])
plt.xlabel("Nhịp (BPM)"); plt.ylabel("Năng lượng"); plt.show()

## Chặng 4: Tự làm
1. Ở ô 7, Lan chưa nghe 'Lá vàng'. Dựa vào bạn có gu giống Lan nhất, em đoán Lan sẽ chấm bài này mấy điểm?
2. Thêm 6 bài hát em thích vào ô 1 (tự ước lượng 3 đặc trưng), chạy lại ô 4 và ô 6. Máy gợi ý có hợp gu em không?
3. Ở ô 5, em chọn k bằng mấy? Vì sao?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*